# 03 - Model Comparison

Tujuan:
- Bandingkan XGBoost, LightGBM, RandomForest
- Tampilkan tabel metrics (RMSE, AUC-ROC, train time)
- Plot confusion matrix, ROC curve, dan feature importance

In [ ]:
from pathlib import Path
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, mean_squared_error
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display

In [ ]:
dataRoot = Path("../data")
dataPath = dataRoot / "processed" / "uss_features.csv"

df = pd.DataFrame()
if dataPath.exists():
    df = pd.read_csv(dataPath)
    display(df.head(3))
else:
    print(f"Missing: {dataPath}")

In [ ]:
if df.empty:
    print("No data loaded.")
else:
    targetCol = "ussLabel"
    if targetCol not in df.columns and "ussScore" in df.columns:
        threshold = 70
        df[targetCol] = (df["ussScore"] >= threshold).astype(int)

    excludeCols = [targetCol]
    featureCols = [c for c in df.columns if c not in excludeCols and df[c].dtype != "O"]

    x = df[featureCols]
    y = df[targetCol]
    print("features:", len(featureCols))

In [ ]:
if not df.empty:
    stratify = y if y.nunique() > 1 else None
    xTrain, xTest, yTrain, yTest = train_test_split(
        x, y, test_size=0.2, random_state=42, stratify=stratify
    )

In [ ]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42)
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9,
        colsample_bytree=0.9, eval_metric="logloss", random_state=42
    )
except Exception as exc:
    print("XGBoost not available:", exc)

try:
    from lightgbm import LGBMClassifier
    models["LightGBM"] = LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=-1, random_state=42
    )
except Exception as exc:
    print("LightGBM not available:", exc)

In [ ]:
results = []
trained = {}
resultsDf = pd.DataFrame()

if df.empty:
    print("No data to train.")
else:
    def evalModel(name, model):
        start = time.time()
        model.fit(xTrain, yTrain)
        trainTime = time.time() - start

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(xTest)[:, 1]
            pred = (proba >= 0.5).astype(int)
            rmse = mean_squared_error(yTest, proba, squared=False)
            auc = roc_auc_score(yTest, proba) if yTest.nunique() > 1 else np.nan
        else:
            pred = model.predict(xTest)
            proba = None
            rmse = mean_squared_error(yTest, pred, squared=False)
            auc = np.nan

        results.append({
            "model": name,
            "rmse": rmse,
            "auc": auc,
            "trainTimeSec": trainTime
        })
        trained[name] = {"model": model, "pred": pred, "proba": proba}

    for name, model in models.items():
        evalModel(name, model)

    resultsDf = pd.DataFrame(results).sort_values(by="auc", ascending=False)
    display(resultsDf)

In [ ]:
if not resultsDf.empty:
    bestName = resultsDf.sort_values(by="auc", ascending=False).iloc[0]["model"]
    best = trained[bestName]
    ConfusionMatrixDisplay(confusion_matrix(yTest, best["pred"])).plot()
    plt.title(f"Confusion matrix - {bestName}")
    plt.show()

    if best["proba"] is not None and yTest.nunique() > 1:
        fpr, tpr, _ = roc_curve(yTest, best["proba"])
        plt.plot(fpr, tpr, label=f"{bestName} (AUC={resultsDf.iloc[0]['auc']:.3f})")
        plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
        plt.xlabel("False positive rate")
        plt.ylabel("True positive rate")
        plt.title("ROC curve")
        plt.legend()
        plt.show()

In [ ]:
if not resultsDf.empty:
    bestName = resultsDf.sort_values(by="auc", ascending=False).iloc[0]["model"]
    bestModel = trained[bestName]["model"]
    if hasattr(bestModel, "feature_importances_"):
        importances = pd.Series(bestModel.feature_importances_, index=featureCols).sort_values(ascending=False)
        display(importances.head(10))
        importances.head(20).sort_values().plot(kind="barh")
        plt.title(f"Feature importance - {bestName}")
        plt.show()
    else:
        print("No feature_importances_ on best model.")

In [ ]:
outDir = dataRoot / "processed"
outDir.mkdir(parents=True, exist_ok=True)

if not resultsDf.empty:
    resultsDf.to_csv(outDir / "model_results.csv", index=False)

if "importances" in locals():
    importances.to_csv(outDir / "feature_importance.csv")